# **Sub Question 1: Which neighborhoods (and census tracts) experienced the highest rates of business closures during 2020-2021, and which show the most new openings (recovery) from 2022 to 2025?**

## Set Up

In [ ]:
!pip install contextily

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

#added more that we use in lab
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive
!ls /content/drive/Shareddrives

In [ ]:
import pandas as pd

gdf = gpd.read_file("/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/cleaned/open_close_after_2016.geojson")

In [ ]:
gdf.info()

In [ ]:
# # adding another plot and importing a shpfile of SF so we can see where the points are in SF

# import matplotlib.pyplot as plt

# # Importing SF geometry
# # URL for 2025 TIGER/Line Place boundaries - info here on
# ## how to use: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]


# project to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

## Plotting Closings vs. Openings over time for all businesses

In [ ]:
import plotly.express as px

# counting number of opening and closings - Abigail
counts = gdf.groupby(["year", "status"]).size().reset_index(name="count")
counts = counts[counts["year"] <= 2025]

# Making graph of this
fig = px.line(
    counts,
    x="year",
    y="count",
    color="status",
    markers=True
)

# Adding labels
fig.update_layout(
    title="San Francisco Business Openings vs Closings (2016–2025)",
    xaxis_title="Year",
    yaxis_title="Number of Businesses",
    legend_title="Status"
)

#adding code to show each year on x-axis
fig.update_xaxes(dtick=1)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

#making background white and removing vert lines
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="lightgray"
)

fig.show()


## **Tract Level Analysis**

In [ ]:
#I added a shp file with census tracts to our folder

tracts_gdf = gpd.read_file(
    "/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/cb_2020_06_tract_500k"
)

In [ ]:
#checking to make sure all good

tracts_gdf.columns

# simplifying
tracts_gdf = tracts_gdf[["NAME", "NAMELSAD", "STATE_NAME","GEOID", "geometry"]]


In [ ]:
# was getting error with "index_right" - removing from both dfs
tracts_gdf = tracts_gdf.drop(columns=["index_right", ], errors="ignore")
gdf = gdf.drop(columns=["index_right"], errors="ignore")

In [ ]:
gdf.info()

In [ ]:
# Joining the gdf and tract gdf so we can summarize within tracts

gdf = gdf.set_crs(epsg=4326)
tracts_gdf = tracts_gdf.to_crs(epsg=4326)

# joining within
business_tracts_gdf = gpd.sjoin(gdf, tracts_gdf, how="left", predicate="within")

In [ ]:
business_tracts_gdf = business_tracts_gdf[(business_tracts_gdf["year"] >= 2016) & (business_tracts_gdf["year"] <= 2025)]

In [ ]:
# I'm counting the number of businesses per tract per year, separated by closing and openings status
tract_year = (
    business_tracts_gdf
    .groupby(["GEOID", "year", "status"])
    .size()
    .reset_index(name="count")
    .pivot(index=["GEOID", "year"], columns="status", values="count")
    .fillna(0)
    .reset_index()
)

In [ ]:
tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
    tract_year,
    on="GEOID",
    how="left"
).fillna(0)

In [ ]:
tracts_plot.info()
tracts_plot = gpd.clip(tracts_plot, sf_poly)

In [ ]:
tracts_plot["year"] = tracts_plot["year"].astype(int)
tracts_plot = tracts_plot.sort_values(["GEOID", "year"])

In [ ]:
print(tracts_plot.columns)

In [ ]:
# opened = tracts_plot[tracts_plot["status"] == "opened"]
# closed = tracts_plot[tracts_plot["status"] == "closed"]

In [ ]:
# This is the code I used to make some interactive visualizations, but the file size was preventing me from uploading to Github. I commented this out for now so I could upload.

# fig_closed = px.choropleth_mapbox(
#     closed,
#     geojson=closed.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="count",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},  # centering on SF becuase it wasn't when I tested
#     color_continuous_scale="Reds",
#     height=600
# )

# fig_closed.update_layout(title="Business Closings by Census Tract (2016–2025)")
# fig_closed.show()

In [ ]:
# fig_opened = px.choropleth_mapbox(
#     opened,
#     geojson=closed.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="count",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},  # centering on SF becuase it wasn't when I tested
#     color_continuous_scale="Blues",
#     height=600
# )

# fig_opened.update_layout(title="Business Openings by Census Tract (2016–2025)")
# fig_opened.show()

Census Tract Analysis - Baselining

In [ ]:
# Reformatting so each row is a unique count per tract, year, and status (so we don't have closings and openings for business in one row)
business_tracts_separated = (
    business_tracts_gdf
    .groupby(["GEOID", "year", "status"])
    .size()
    .reset_index(name="count")
)

In [ ]:
# Baselining by 2016

baseline = business_tracts_separated[business_tracts_separated["year"] == 2016][["GEOID", "status", "count"]].rename(columns={"count": "baseline"}) # makes new df and filtering for only 2016 so we can use as baseline
business_tracts_separated = business_tracts_separated.merge(baseline, on=["GEOID", "status"], how="left") # joining this new baseline df to business tracts df so we can calculate change
business_tracts_separated["change_from_2016"] = business_tracts_separated["count"] - business_tracts_separated["baseline"] # subtracts change from baseline


In [ ]:
tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
    business_tracts_separated,
    on="GEOID",
    how="left"
).fillna(0)

In [ ]:
opened = tracts_plot[tracts_plot["status"] == "opened"]
closed = tracts_plot[tracts_plot["status"] == "closed"]

In [ ]:
# sanity check on changes

opened["change_from_2016"].describe()

# most tracts didn't have major changes (average of 3 fewer openings compared to 2016)
# but some tracts had a major change (both pos and neg compared to 2016)- these extremes will skew our viz

In [ ]:
closed["change_from_2016"].describe()

In [ ]:
# fig = px.choropleth_mapbox(
#     opened,
#     geojson=tracts_plot.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="change_from_2016",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=11,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="RdYlGn",
#     range_color=[-20, 20], # adding this so that the legend scale doesn't change each year - need to make decision around this bc min and max will not show most changes
#     height=600,
# )

# fig.update_layout(title="SF Business Openings Change from 2016 by Census Tract")
# fig.show()

In [ ]:
# fig.write_html("sf_business_map.html")

In [ ]:
# from google.colab import files
# files.download("sf_business_map.html")

In [ ]:
# fig = px.choropleth_mapbox(
#     closed,
#     geojson=tracts_plot.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="change_from_2016",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=11,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="RdYlGn",
#     range_color=[-10,20],
#     height=600,
# )

# fig.update_layout(title="SF Business Closings Change from 2016 by Census Tract")
# fig.show()

## Neighborhood Level Analysis

In [ ]:
# Adding SF neighborhood geometry

nbhd_gdf = gpd.read_file(
    "/content/drive/MyDrive/Courses/Spring 2026/Urban Informatics/C255_final_project/raw/SF_neighborhoods_geom"
)

In [ ]:
nbhd_gdf.drop(columns=["link"], inplace=True)

In [ ]:
nbhd_gdf.columns

In [ ]:
# Joining the gdf and nbhd gdf so we can summarize within nbhds

gdf = gdf.set_crs(epsg=4326)
nbhd_gdf = nbhd_gdf.to_crs(epsg=4326)

# joining within
business_neighborhood_gdf = gpd.sjoin(gdf, nbhd_gdf, how="left", predicate="within")

In [ ]:
business_neighborhood_gdf.columns

Which neighborhoods (and census tracts) experienced the highest rates of business closures during 2020-2021, and which show the most new openings (recovery) from 2022 to 2025?

In [ ]:
gdf = business_neighborhood_gdf.copy()

# filtering for 2019 to 2025
gdf = gdf[gdf['year'].between(2019, 2025)]

# openings
openings = (
    gdf[gdf['status'] == 'opened']
    .groupby(['name', 'year'])
    .size()
    .unstack(fill_value=0)
)

# closings
closings = (
    gdf[gdf['status'] == 'closed']
    .groupby(['name', 'year'])
    .size()
    .unstack(fill_value=0)
)

# making sure all years are there
all_years = range(2019, 2026)
openings = openings.reindex(columns=all_years, fill_value=0)
closings = closings.reindex(columns=all_years, fill_value=0)

# renaming columns to be open_year
openings.columns = [f"open_{y}" for y in openings.columns]
closings.columns = [f"close_{y}" for y in closings.columns]

# combining openings and closings to get a table to compare across neighborhoods
neighborhood_table = openings.join(closings, how='outer').fillna(0).astype(int)

neighborhood_table

Need to baseline by number of businesses in each neighborhood

In [ ]:
gdf = business_neighborhood_gdf.copy()

# baseline by total # of businesses
baseline = gdf.groupby('name')['uniqueid'].nunique()

# closures during COVID (2020–2021)
closures = gdf[
    (gdf['year'].between(2020, 2021)) &
    (gdf['status'] == 'closed')
].groupby('name').size()

# openings during recovery (2022–2025)
openings = gdf[
    (gdf['year'].between(2022, 2025)) &
    (gdf['status'] == 'opened')
].groupby('name').size()

# combine
rate_table = pd.DataFrame({
    'baseline': baseline,
    'closures_2020_21': closures,
    'openings_2022_25': openings
}).fillna(0)

# adding rates
rate_table['closure_rate'] = rate_table['closures_2020_21'] / rate_table['baseline']
rate_table['recovery_rate'] = rate_table['openings_2022_25'] / rate_table['baseline']

rate_table

## Business Resilience Rate

In [ ]:
df = rate_table_filtered.copy()
df['resilience'] = df['recovery_rate'] - df['closure_rate']

top_10 = df.sort_values('resilience', ascending=False).head(10)
bottom_10 = df.sort_values('resilience', ascending=True).head(10)

print("Most Resilient Neighborhoods")
display(top_10)

print("Least Resilient Neighborhoods")
display(bottom_10)

In [ ]:
# import ipywidgets as widgets

# years = list(range(2019, 2026))

# def plot_neighborhood(neighborhood_name):

#     # baselining data from nbhd table
#     row = neighborhood_table.loc[neighborhood_name]
#     base = rate_table.loc[neighborhood_name, 'baseline']

#     openings = row[[f"open_{y}" for y in years]] / base
#     closings = row[[f"close_{y}" for y in years]] / base

#     # make closings negative
#     closings = -closings

#     # plotting bar graph
#     plt.figure(figsize=(8,4))

#     plt.bar(years, openings, label='Openings (rate)', color='blue')
#     plt.bar(years, closings, label='Closings (rate)', color='orange')

#     plt.xlabel('Year')
#     plt.ylabel('Rate (normalized)')
#     plt.title(f'Business Open/Close Rates: {neighborhood_name}')

#     plt.xticks(years)
#     plt.ylim(-0.2, 0.2)

#     plt.legend()
#     plt.tight_layout()
#     plt.show()


# widgets.interact(
#     plot_neighborhood,
#     neighborhood_name=widgets.Dropdown(options=neighborhood_table.index)
# )

Now plotting businesses closure rate by neighborhood

In [ ]:
# Filtering to nbhds with at least 50 businesses
rate_table_filtered = rate_table[rate_table['baseline'] >= 50]

In [ ]:
# import plotly.express as px

# map_gdf = nbhd_gdf.copy()

# # merge closure rates
# map_gdf = map_gdf.merge(
#     rate_table_filtered[['closure_rate']],
#     left_on='name',
#     right_index=True,
#     how='left'
# )

# fig = px.choropleth_mapbox(
#     map_gdf,
#     geojson=map_gdf.__geo_interface__,
#     locations=map_gdf.index,
#     color="closure_rate",
#     hover_name="name",
#     hover_data={"closure_rate": True},
#     color_continuous_scale="Reds",
#     center={"lat": 37.77, "lon": -122.42},
#     mapbox_style="carto-positron",
#     zoom=11,
#     opacity=0.7
# )

# fig.update_layout(
#     title="Business COVID Closure Rate (2020–2022)",
#     margin={"r":0,"t":40,"l":0,"b":0}
# )

# fig.show()

In [ ]:
# # merge opening rates
# map_gdf = map_gdf.merge(
#     rate_table_filtered[['recovery_rate']],
#     left_on='name',
#     right_index=True,
#     how='left'
# )

# fig = px.choropleth_mapbox(
#     map_gdf,
#     geojson=map_gdf.__geo_interface__,
#     locations=map_gdf.index,
#     color="recovery_rate",
#     hover_name="name",
#     hover_data={"recovery_rate": True},
#     color_continuous_scale="Greens",
#     center={"lat": 37.77, "lon": -122.42},
#     mapbox_style="carto-positron",
#     zoom=11,
#     opacity=0.7
# )

# fig.update_layout(
#     title="Business Recovery Rate (2022–2025)",
#     margin={"r":0,"t":40,"l":0,"b":0}
# )

# fig.show()

In [ ]:
# import matplotlib.pyplot as plt

# # Get top & bottom 10
# top_10 = df.sort_values('resilience', ascending=False).head(10)
# bottom_10 = df.sort_values('resilience', ascending=True).head(10)

# # Scatterplot
# plt.figure(figsize=(8,8))
# plt.scatter(df['closure_rate'], df['recovery_rate'], alpha=0.6)

# # Label top 10
# for name, row in top_10.iterrows():
#     plt.text(row['closure_rate'], row['recovery_rate'], name, fontsize=8)

# # Label bottom 10
# for name, row in bottom_10.iterrows():
#     plt.text(row['closure_rate'], row['recovery_rate'], name, fontsize=8)


# plt.xlabel('Closure Rate (2020–2021)')
# plt.ylabel('Recovery Rate (2022–2025)')
# plt.title('COVID Impact vs Recovery')

# plt.axvline(df['closure_rate'].mean(), linestyle='--', color='gray')
# plt.axhline(df['recovery_rate'].mean(), linestyle='--', color='gray')

# plt.show()

In [ ]:
# import plotly.graph_objects as go

# df = df.copy()
# neighborhoods = df.index.tolist()

# base_color = "lightgray"
# highlight_color = "red"

# x_mean = df["closure_rate"].mean()
# y_mean = df["recovery_rate"].mean()

# fig = go.Figure()

# # Scatter
# fig.add_trace(
#     go.Scatter(
#         x=df["closure_rate"],
#         y=df["recovery_rate"],
#         mode="markers",
#         text=df.index,
#         hovertemplate=
#             "<b>%{text}</b><br>" +
#             "Closure rate: %{x:.3f}<br>" +
#             "Recovery rate: %{y:.3f}<br>" +
#             "<extra></extra>",
#         marker=dict(color=base_color, size=10),
#     )
# )

# # Quadrant lines
# fig.add_shape(
#     type="line",
#     x0=x_mean, x1=x_mean,
#     y0=df["recovery_rate"].min(),
#     y1=df["recovery_rate"].max(),
#     line=dict(dash="dash", color="black")
# )

# fig.add_shape(
#     type="line",
#     x0=df["closure_rate"].min(),
#     x1=df["closure_rate"].max(),
#     y0=y_mean,
#     y1=y_mean,
#     line=dict(dash="dash", color="black")
# )

# # Dropdown
# buttons = []

# # reset
# buttons.append(
#     dict(
#         label="All",
#         method="update",
#         args=[
#             {"marker": {"color": [base_color] * len(neighborhoods)}},
#             {"title": "All Neighborhoods"}
#         ],
#     )
# )

# # highlight each neighborhood
# for n in neighborhoods:
#     colors = [highlight_color if name == n else base_color for name in neighborhoods]

#     buttons.append(
#         dict(
#             label=n,
#             method="update",
#             args=[
#                 {"marker": {"color": colors}},
#                 {"title": f"Highlight: {n}"}
#             ],
#         )
#     )

# fig.update_layout(
#     title="COVID Impact vs Recovery",
#     xaxis_title="Closure Rate (2020–2021)",
#     yaxis_title="Recovery Rate (2022–2025)",
#     updatemenus=[dict(buttons=buttons, direction="down", showactive=True)]
# )

# fig.show()